# Gaussian white-noise realizations

Shows how the noise realizations of the dataset are generated from the
tabulated standard-deviation maps in `maps/Noise/SIGMA_<channel>.fits`.

The noise is Gaussian and white; 1/f and atmospheric noise are not included
in this dataset.  The noise maps are drawn at HEALPix `Nside = 1024` in
thermodynamic units (`uK_CMB`) and should be masked when they are used.


In [ ]:
import os
import pathlib
import sys

import numpy as np
import healpy as hp
from numpy.random import default_rng

PKG = pathlib.Path('..').resolve()                 # repository root
if not (PKG / 'deproj_utils.py').exists():
    PKG = pathlib.Path('.').resolve()
sys.path.insert(0, str(PKG))
from deproj_utils import plot_maps

DATA = pathlib.Path(os.environ.get('DEPROJ_DATA_ROOT', '/sharefs/alicpt/users/doujiazheng/data/deproj'))
OUT = pathlib.Path('output')
OUT.mkdir(exist_ok=True)

NSIDE = 1024

## White-noise generator

`SIGMA_<channel>.fits` holds the I, Q and U standard deviation per pixel.  The
WMAP K band is provided at a lower resolution and is rescaled, conserving the
variance per unit solid angle, before drawing the noise.


In [ ]:
def sigma_maps(channel, nside=NSIDE):
    sigma = hp.read_map(DATA / 'maps' / 'Noise' / ('SIGMA_%s.fits' % channel),
                        field=None, dtype=np.float64)
    nside_in = hp.get_nside(sigma[0])
    if nside_in == nside:
        return sigma
    x, y, z = hp.pix2vec(nside, np.arange(hp.nside2npix(nside)))
    return sigma[:, hp.vec2pix(nside_in, x, y, z)] * (nside / nside_in)


def gaussian_noise(channel, nside=NSIDE, seed=None):
    """One white-noise IQU realization, in uK_CMB."""
    rng = default_rng(seed)
    return rng.standard_normal((3, hp.nside2npix(nside))) * sigma_maps(channel, nside)

In [ ]:
noise = gaussian_noise('ALI_150', seed=0)

plot_maps([noise[i] for i in range(3)], titles=['I', 'Q', 'U'],
          vmin=-20., vmax=20., unit=r'$\mu$K',
          outfile=OUT / 'noise_ALI_150.png');

`noise` is an `(3, npix)` array.  The other bands of the dataset are
`ALI_95`, `HFI_100`, `HFI_143`, `HFI_217`, `HFI_353` and `WMAP_K`, and any
integer can be used as `seed`.  To store a realization use

```python
hp.write_map('noise_ALI_150.fits', noise['ALI_150'], overwrite=True, dtype=np.float64)
```

and remember to multiply the noise by `mask` when coadding it with a sky map.
